# LG-CXR FRCNN Kaggle V2: Three-Model Cascade

This is the complete V2 Kaggle workflow with three models:

1. Faster R-CNN scanner/detector
2. ResNet18 global image classifier
3. ResNet18 crop verifier classifier

The final submission is produced after score fusion.

## 1. Clone Or Update Repo

In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/thomas240103/Amia_keagle_challenge.git'
PROJECT_DIR = Path('/kaggle/working/Amia_keagle_challenge')

if PROJECT_DIR.exists():
    !git -C /kaggle/working/Amia_keagle_challenge pull --ff-only
else:
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', Path.cwd())

## 2. Install Minimal Extras

In [ ]:
import importlib.util

missing = []
if importlib.util.find_spec('yaml') is None:
    missing.append('PyYAML')
if importlib.util.find_spec('timm') is None:
    missing.append('timm')

print('Missing optional packages:', missing)
if 'PyYAML' in missing:
    !pip install -q PyYAML
if 'timm' in missing:
    !pip install -q --no-deps timm

print('Done. Do not run pip install -r requirements.txt on Kaggle; it may disturb Kaggle CUDA/RAPIDS packages.')

## 3. Check GPU And Paths

In [ ]:
import os
from pathlib import Path
import torch, torchvision

CONFIG = 'configs/v2_three_model.yaml'
DATA_ROOT = '/kaggle/input/amia-public-challenge-2026'
WORK_DIR = '/kaggle/working'
os.environ['LGCXR_DATA_ROOT'] = DATA_ROOT
os.environ['LGCXR_WORK_DIR'] = WORK_DIR
os.environ['LGCXR_ALLOW_WEIGHT_DOWNLOAD'] = '1'  # Use pretrained torchvision detector weights when Kaggle internet is enabled.

print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('DATA_ROOT exists:', Path(DATA_ROOT).exists())
for name in ['train', 'test', 'train.csv', 'test.csv', 'img_size.csv', 'sample_submission.csv']:
    print(name, (Path(DATA_ROOT) / name).exists())

## 4. CPU-Safe Checks

Run these before spending GPU. They do not train models.

In [ ]:
!python scripts/06_ci_checks.py
!python scripts/00_preflight.py --config {CONFIG}
!python scripts/05_audit_dimensions.py --config {CONFIG}

## 5. Optional V2 Smoke Test

This still trains models, so skip it if GPU budget is tight. It is useful only to check the whole cascade.

In [ ]:
# !python scripts/09_full_pipeline_v2.py --config {CONFIG} --smoke-test --force-train

## 6. Model 1: Train Faster R-CNN Scanner

In [ ]:
!python scripts/01_train_scanner.py --config {CONFIG} --skip-preflight

## 7. Inspect Scanner Metrics Before Continuing

If the scanner mAP proxy is near zero after several epochs, fix scanner training before spending GPU on the global/crop classifiers.

In [ ]:
import json
from pathlib import Path

summary_path = Path('/kaggle/working/lgcxr_scanner_training_summary.json')
print('summary exists:', summary_path.exists())
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('best_score:', summary.get('best_score'))
    for row in summary.get('history', [])[-5:]:
        print({k: row.get(k) for k in ['epoch', 'train_loss', 'val_map_proxy']})
    last_metrics = summary.get('history', [{}])[-1].get('metrics', {})
    print('last validation mAP proxy:', last_metrics.get('map'))
    print('last per-class AP:', last_metrics.get('per_class_ap'))

## 8. Scanner Prediction

In [ ]:
!python scripts/02_predict_scanner.py --config {CONFIG}

## 9. Model 2: Train Global ResNet18 Classifier

This model sees the whole X-ray and predicts which classes are likely anywhere in the image.

In [ ]:
!python scripts/07_train_global_classifier.py --config {CONFIG}

## 10. Global Classifier Prediction

In [ ]:
!python scripts/10_predict_global_classifier.py --config {CONFIG}

## 11. Model 3: Train Crop ResNet18 Classifier

This model sees ground-truth pathology crops during training and later verifies Faster R-CNN candidate boxes.

In [ ]:
!python scripts/08_train_crop_classifier.py --config {CONFIG}

## 12. Crop Classifier Prediction

In [ ]:
!python scripts/11_predict_crop_classifier.py --config {CONFIG}

## 13. Fuse Three Model Scores

In [ ]:
!python scripts/12_fuse_predictions_v2.py --config {CONFIG}

## 14. Make Final V2 Submission

In [ ]:
!python scripts/03_make_submission.py --config {CONFIG} --prediction-csv /kaggle/working/lgcxr_fused_test_predictions.csv

## 15. Inspect Submission

In [ ]:
import pandas as pd
from pathlib import Path
p = Path('/kaggle/working/submission.csv')
print('exists:', p.exists())
if p.exists():
    sub = pd.read_csv(p)
    print(sub.shape)
    display(sub.head())

## 16. One-Command V2 Pipeline

Optional alternative after you trust the setup.

In [ ]:
# !python scripts/09_full_pipeline_v2.py --config {CONFIG} --force-train